# 04 — Resultados del pipeline clásico (H4 + H5)

Inspección visual del modelo entrenado: detecciones sobre test, conteos por clase, distribución de scores, ejemplos de aciertos y fallos.

**Requiere haber corrido `scripts/run_classical_train.py`** (y opcionalmente `run_classical_hard_neg.py` + `run_classical_infer.py`). Si no, el notebook avisa al cargar el modelo y para ahí.

Las métricas formales completas (mAP@0.5, F1 por clase, matriz de confusión 20×20) son entregable de **H7 — framework de evaluación común**, donde se comparan también contra YOLO sobre el mismo test.

In [ ]:
import json
import sys
from pathlib import Path

notebook_dir = Path.cwd()
repo_root = notebook_dir.parent if notebook_dir.name == "notebooks" else notebook_dir
sys.path.insert(0, str(repo_root / "src"))

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from grocery_detection.utils.config import load_yaml
from grocery_detection.data.eda import load_coco
from grocery_detection.data.visualize import draw_bbox
from grocery_detection.classical.descriptors.bovw import load_codebook
from grocery_detection.classical.classifier import load as load_classifier
from grocery_detection.classical.pipeline import detect
from grocery_detection.classical.iou import iou_matrix

data_cfg = load_yaml(repo_root / "configs" / "data.yaml")
cls_cfg = load_yaml(repo_root / "configs" / "classical.yaml")
classes_cfg = load_yaml(repo_root / "configs" / "classes.yaml")

test_coco = load_coco(repo_root / data_cfg["filtered_splits"]["test"])
img_dir = repo_root / data_cfg["paths"]["d2s_images"]
model_path = repo_root / cls_cfg["paths"]["model"]
codebook_path = repo_root / cls_cfg["paths"]["codebook"]
predictions_path = repo_root / cls_cfg["paths"]["predictions"]

target_names = [n for g in classes_cfg["target_classes"] for n in g["items"]]
id2name = {i + 1: n for i, n in enumerate(target_names)}
print(f"Test imgs    : {len(test_coco['images'])}")
print(f"Model        : {model_path}  (exists={model_path.exists()})")
print(f"Codebook     : {codebook_path}  (exists={codebook_path.exists()})")
print(f"Predictions  : {predictions_path}  (exists={predictions_path.exists()})")

In [ ]:
if not model_path.exists():
    raise FileNotFoundError(
        f"Modelo no entrenado todavía: {model_path}.\n"
        "Ejecuta primero (overnight):\n"
        "  .\\scripts\\run_overnight.ps1\n"
        "o al menos: uv run python scripts/run_classical_train.py"
    )

classifier = load_classifier(model_path)
codebook = load_codebook(codebook_path)
print(f"Classifier cargado. Target classes: {len(classifier.target_class_ids)}")

## 1. Detecciones sobre 6 imágenes test (side-by-side con GT)

Imágenes test reales con escenas cluttered (varios productos). Izquierda = ground truth; derecha = predicción del modelo.

In [ ]:
import random
rng = random.Random(0)

anns_by_img = {}
for a in test_coco["annotations"]:
    anns_by_img.setdefault(a["image_id"], []).append(a)
id2img = {im["id"]: im for im in test_coco["images"]}

# Escenas con >= 3 GT objetos -> visualmente más interesantes
candidates = [iid for iid, anns in anns_by_img.items() if len(anns) >= 3]
sample_ids = rng.sample(candidates, min(6, len(candidates)))

inf_cfg = cls_cfg["inference"]
prop_cfg = cls_cfg["proposals"]

fig, axes = plt.subplots(len(sample_ids), 2, figsize=(14, 6 * len(sample_ids)))
for row, iid in enumerate(sample_ids):
    info = id2img[iid]
    img_bgr = cv2.imread(str(img_dir / info["file_name"]))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # GT panel
    gt_img = img_rgb.copy()
    for a in anns_by_img[iid]:
        x, y, w, h = a["bbox"]
        draw_bbox(gt_img, x, y, w, h, label=id2name[a["category_id"]], color=(0, 200, 0))

    # Predict panel
    dets = detect(
        img_bgr, classifier=classifier, codebook=codebook,
        proposals_mode=prop_cfg["mode"],
        proposals_max_per_image=prop_cfg["max_per_image"],
        proposals_max_side=prop_cfg["max_side"],
        score_thresh=inf_cfg["score_thresh"],
        nms_iou=inf_cfg["nms_iou"],
        top_k=inf_cfg["top_k_per_image"],
    )
    pred_img = img_rgb.copy()
    for d in dets[:15]:  # show up to 15 detections per image
        x, y, w, h = d.bbox
        draw_bbox(pred_img, x, y, w, h,
                  label=f"{id2name[d.class_id]} {d.score:.2f}",
                  color=(180, 50, 240))

    axes[row][0].imshow(gt_img); axes[row][0].set_title(f"GT ({info['file_name']}): {len(anns_by_img[iid])} obj."); axes[row][0].axis("off")
    axes[row][1].imshow(pred_img); axes[row][1].set_title(f"Predicciones: {len(dets)} (top 15 mostradas)"); axes[row][1].axis("off")
plt.tight_layout(); plt.show()

## 2. Conteo de detecciones por clase (sobre las 6 imágenes anteriores)

Mira si el modelo predice de manera balanceada o si se sesga a alguna clase concreta.

In [ ]:
# Recoger todas las detecciones de las 6 imgs en una pasada agregada
all_dets = []
for iid in sample_ids:
    info = id2img[iid]
    img_bgr = cv2.imread(str(img_dir / info["file_name"]))
    dets = detect(
        img_bgr, classifier=classifier, codebook=codebook,
        proposals_mode=prop_cfg["mode"],
        proposals_max_per_image=prop_cfg["max_per_image"],
        proposals_max_side=prop_cfg["max_side"],
        score_thresh=inf_cfg["score_thresh"],
        nms_iou=inf_cfg["nms_iou"],
        top_k=inf_cfg["top_k_per_image"],
    )
    for d in dets:
        all_dets.append({"class": id2name[d.class_id], "score": d.score})
df_dets = pd.DataFrame(all_dets)
if df_dets.empty:
    print("Sin detecciones por encima del threshold. Revisa score_thresh en classical.yaml.")
else:
    counts = df_dets["class"].value_counts()
    counts_full = pd.Series(0, index=target_names)
    counts_full.update(counts)
    ax = counts_full.plot.barh(figsize=(8, 6), color="#9333EA")
    ax.set_xlabel("# detecciones")
    ax.set_title("Detecciones por clase (6 imgs test)")
    ax.invert_yaxis()
    plt.tight_layout(); plt.show()

## 3. Distribución de scores

In [ ]:
if not df_dets.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(df_dets["score"], bins=30, color="#9333EA", alpha=0.7, edgecolor="white")
    ax.axvline(inf_cfg["score_thresh"], color="#DC2626", linestyle="--",
               label=f"score_thresh = {inf_cfg['score_thresh']}")
    ax.set_xlabel("score (decision_function)")
    ax.set_ylabel("# detecciones")
    ax.set_title("Distribución de confianzas de las detecciones")
    ax.legend()
    plt.tight_layout(); plt.show()
    print(df_dets["score"].describe())

## 4. Si tienes el JSON de inferencia completa, estadísticas globales

Generado por `scripts/run_classical_infer.py` en el overnight. Si todavía no existe, esta sección se salta automáticamente.

In [ ]:
if not predictions_path.exists():
    print(f"Aún no hay JSON de inferencia completa en {predictions_path}.")
    print("Ejecuta: uv run python scripts/run_classical_infer.py")
else:
    with open(predictions_path, encoding="utf-8") as f:
        preds = json.load(f)
    print(f"Total predicciones sobre test: {len(preds)}")
    dfp = pd.DataFrame(preds)
    print("Score stats:", dfp["score"].describe().to_dict())
    by_class = dfp["category_id"].map(id2name).value_counts()
    by_class_full = pd.Series(0, index=target_names)
    by_class_full.update(by_class)
    ax = by_class_full.plot.barh(figsize=(8, 6), color="#6B21A8")
    ax.set_xlabel("# predicciones"); ax.set_title("Predicciones por clase — test completo (1407 imgs)"); ax.invert_yaxis()
    plt.tight_layout(); plt.show()

## 5. Confusión rápida vs GT (sobre el test completo)

Heurística simple: cada predicción se empareja con su GT más solapado si IoU ≥ 0.5. Eso da una matriz 20×20 que ilustra dónde el clásico confunde clases. Las métricas formales (mAP, F1) son entregable de H7.

In [ ]:
if predictions_path.exists():
    test_anns_by_img = {}
    for a in test_coco["annotations"]:
        test_anns_by_img.setdefault(a["image_id"], []).append(a)
    preds_by_img = {}
    for p in preds:
        preds_by_img.setdefault(p["image_id"], []).append(p)

    K = len(target_names)
    confusion = np.zeros((K + 1, K + 1), dtype=np.int64)  # +1 for "no match"

    iou_thresh = 0.5
    for iid, gts in test_anns_by_img.items():
        gt_boxes = np.array([g["bbox"] for g in gts], dtype=np.float32)
        gt_labels = np.array([g["category_id"] for g in gts], dtype=np.int64)
        preds_here = preds_by_img.get(iid, [])
        if not preds_here:
            continue
        pred_boxes = np.array([p["bbox"] for p in preds_here], dtype=np.float32)
        pred_labels = np.array([p["category_id"] for p in preds_here], dtype=np.int64)
        ious = iou_matrix(pred_boxes, gt_boxes)
        used_gt = set()
        # Orden por confianza descendente para greedy matching
        scores = np.array([p["score"] for p in preds_here])
        order = np.argsort(-scores)
        for i in order:
            j_best = int(np.argmax(ious[i])) if ious.shape[1] else -1
            if j_best >= 0 and ious[i, j_best] >= iou_thresh and j_best not in used_gt:
                used_gt.add(j_best)
                confusion[gt_labels[j_best] - 1, pred_labels[i] - 1] += 1
            else:
                # falso positivo: pred sin GT match
                confusion[K, pred_labels[i] - 1] += 1
        # GTs no matcheadas -> falsos negativos
        for j, _ in enumerate(gt_labels):
            if j not in used_gt:
                confusion[gt_labels[j] - 1, K] += 1

    labels_axis = target_names + ["<none>"]
    fig, ax = plt.subplots(figsize=(10, 9))
    im = ax.imshow(confusion, cmap="Purples")
    ax.set_xticks(range(K + 1)); ax.set_xticklabels(labels_axis, rotation=90, fontsize=7)
    ax.set_yticks(range(K + 1)); ax.set_yticklabels(labels_axis, fontsize=7)
    ax.set_xlabel("predicción"); ax.set_ylabel("ground truth")
    ax.set_title(f"Matriz de confusión heurística (IoU>={iou_thresh})")
    for i in range(K + 1):
        for j in range(K + 1):
            v = confusion[i, j]
            if v:
                ax.text(j, i, str(v), ha="center", va="center", fontsize=6,
                        color="white" if v > confusion.max() * 0.5 else "black")
    plt.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout(); plt.show()

    # Accuracy global (sobre GTs matcheadas)
    diag = np.trace(confusion[:K, :K])
    total_gt = confusion.sum() - confusion[K, :].sum()  # excluye FPs sin GT
    print(f"Aciertos GT->correcto: {diag} / {total_gt} = {diag/total_gt:.1%} (heurística)")

## Conclusiones para enseñar

Lo que se ve en este notebook:

- **Sección 1**: el modelo clásico produce detecciones plausibles sobre escenas cluttered reales.
- **Sección 2-3**: distribución de clases predichas y de confianzas — si está balanceada o sesgada.
- **Sección 4**: volumen total de predicciones sobre el test completo (1407 imágenes).
- **Sección 5**: matriz de confusión heurística — dónde el clásico se confunde. **Esperable**: confusiones dentro de grupos (las 4 manzanas entre sí, las 3 pastas Reggia, las 2 Coca-Cola) y muchas filas/columnas en "<none>" (falsos negativos y falsos positivos).

**Comparativa formal contra YOLOv8s** sobre los mismos test, mismas métricas oficiales (mAP@0.5, F1 macro/por clase, IoU medio, tiempos): **H7**.